# 1. Preprocess claim documents and prepare index-ready rows
Prepare claim evidence: read files and email attachments, review analyzer routing, extract native content, create contextual content embeddings, preserve citations, and save index-ready rows. Previews never print full records or vectors.

Use the root `.venv` and `.env`. Everything is inline. This notebook does **not** create or load a Search index. Photo descriptions are visual observations, not findings of cause, timing, liability or coverage.

In [1]:
import os, json, hashlib, time
from pathlib import Path, PureWindowsPath
import pandas as pd
import extract_msg
from extract_msg.msg_classes import MSGFile
import tiktoken
from dotenv import load_dotenv, find_dotenv
from IPython.display import Markdown, display
from langchain_text_splitters import MarkdownHeaderTextSplitter
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.core.serialization import AzureJSONEncoder
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import AnalysisInput

ENV_FILE = Path(find_dotenv(usecwd=True, raise_error_if_not_found=True))
load_dotenv(ENV_FILE, override=True)
ROOT = ENV_FILE.parent
claim_folder = ROOT / os.getenv('CLAIM_FOLDER', r'data\claims\claim-001')
claim_id = claim_folder.name
output_root = ROOT / os.getenv('CLAIMS_OUTPUT_ROOT', r'output\claims')
output_folder = output_root / claim_id
output_folder.mkdir(parents=True, exist_ok=True)
(output_folder / 'attachments').mkdir(exist_ok=True)
results_folder = output_root / 'cu-results'
results_folder.mkdir(exist_ok=True)
api_version = os.getenv('CU_API_VERSION', '2025-11-01')
document_analyzer_id = 'prebuilt-documentSearch'
image_analyzer_id = 'prebuilt-imageSearch'
credential = DefaultAzureCredential()
cu_client = ContentUnderstandingClient(endpoint=os.environ['APP_CONTENT_UNDERSTANDING_ENDPOINT'], credential=credential, api_version=api_version)
azure_openai_embedding_deployment = os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT') or os.getenv('EMBEDDING_MODEL_DEPLOYMENT')
azure_openai_model_dimensions = int(os.getenv('AZURE_OPENAI_EMBEDDING_DIMENSIONS', '3072'))
print('Claim:', claim_id)
print('Input:', claim_folder)
print('Output:', output_folder)
print('Documents:', document_analyzer_id, '| Images:', image_analyzer_id, '| API:', api_version)

Claim: claim-001
Input: c:\Users\jomedin\Documents\azure-ai-foundry-insurance\data\claims\claim-001
Output: c:\Users\jomedin\Documents\azure-ai-foundry-insurance\output\claims\ais-foundry-eval-dev-02\claim-001
Documents: prebuilt-documentSearch | Images: prebuilt-imageSearch | API: 2025-11-01


## Compact display settings
Run just the next cell to apply small previews in an already-running kernel. It does not change data or call Azure.

In [2]:
pd.set_option('display.max_rows', 5)
pd.set_option('display.max_columns', 15)
pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_seq_items', 3)

def ShowPreview(frame):
    preview = frame.head(5).copy()
    for column in preview.columns:
        if column.endswith('_vector'):
            preview[column] = preview[column].map(lambda vector: f'{len(vector)} dimensions')
        else:
            preview[column] = preview[column].map(lambda value: str(value)[:160])
    display(preview)

## Local syntax and schema checks
Check the actual code cells and the shared evidence-column contract before making Azure calls. This only reads the three notebooks; it does not execute another notebook or change an index.

In [3]:
import ast
notebook_columns = []
for notebook_name in ['01_cu_extraction_step_by_step.ipynb', '02_create_index.ipynb', '03_populate_index.ipynb']:
    notebook = json.loads((ROOT / 'demo' / 'rag' / 'notebooks' / notebook_name).read_text(encoding='utf-8'))
    code = '\n\n'.join(''.join(cell['source']) for cell in notebook['cells'] if cell['cell_type'] == 'code')
    tree = ast.parse(code, filename=notebook_name)
    assignment_name = 'fields' if notebook_name.startswith('02_') else 'index_columns'
    assignment = next(node for node in tree.body if isinstance(node, ast.Assign) and any(isinstance(target, ast.Name) and target.id == assignment_name for target in node.targets))
    if assignment_name == 'fields':
        columns = [ast.literal_eval(next(keyword.value for keyword in call.keywords if keyword.arg == 'name')) for call in assignment.value.elts]
    else:
        columns = ast.literal_eval(assignment.value)
    notebook_columns.append(columns)
if not notebook_columns[0] == notebook_columns[1] == notebook_columns[2] or len(set(notebook_columns[0])) != len(notebook_columns[0]):
    raise ValueError('Preprocessing, index schema and loader columns differ or contain duplicates.')
print('Local checks passed: 3 notebooks parse; all share', len(notebook_columns[0]), 'evidence fields.')

Local checks passed: 3 notebooks parse; all share 18 evidence fields.


## Step 1: Read and display the source files
Only the selected claim folder is read. Review aids outside it are not inputs.

In [4]:
print('-' * 72)
print('Step 1: Reading claim files')
files = sorted(path for path in claim_folder.rglob('*') if path.is_file())
if not files:
    raise ValueError(f'No files found in {claim_folder}')
file_df = pd.DataFrame([{'filename': path.name, 'extension': path.suffix.lower(), 'bytes': path.stat().st_size, 'path': str(path)} for path in files])
ShowPreview(file_df[['filename', 'extension', 'bytes']])
print('Source files:', len(file_df))

------------------------------------------------------------------------
Step 1: Reading claim files


,filename,extension,bytes
0,01-notification.msg,.msg,90624
1,02-inspection-and-attachments.msg,.msg,414720
2,03-adjuster-follow-up.msg,.msg,16896
3,claim-form.pdf,.pdf,1813
4,damage-01.jpg,.jpg,49288


Source files: 10


## Step 2: Read MSG text and attachments locally
MSG files themselves are not sent to CU. Identical PDF/image bytes share one document entry, while all source names and parent-email links remain available. Nested MSGs are parsed locally too. The fixture's named signature PNG is branding, not a damage photo.

In [5]:
documents, emails, skipped = {}, {}, []

def AddDocument(data, filename, source, parent_email=None):
    extension = Path(filename).suffix.lower()
    if extension not in {'.pdf', '.jpg', '.jpeg', '.png'}:
        raise ValueError(f'Unsupported attachment: {source}')
    if filename.casefold() == 'email-signature.png':
        skipped.append({'source': source, 'reason': 'fixture signature/branding'})
        return
    document_hash = hashlib.sha256(data).hexdigest()
    evidence_path = output_folder / 'attachments' / f'{document_hash}{extension}' if parent_email else claim_folder.joinpath(*PureWindowsPath(source).parts[1:])
    item = documents.setdefault(document_hash, {'data': data, 'filename': filename, 'source_path': str(evidence_path.resolve()), 'source_type': 'document' if extension == '.pdf' else 'image', 'source_files': [], 'parent_emails': []})
    item['source_files'].append(source)
    if parent_email and parent_email not in item['parent_emails']:
        item['parent_emails'].append(parent_email)
    if parent_email:
        (output_folder / 'attachments' / f'{document_hash}{extension}').write_bytes(data)

def ReadMessage(message, source, parent_email=None):
    if message.body is None and message.htmlBody:
        raise ValueError(f'MSG has HTML but no parsed text body: {source}. Inspect it before dropping its body.')
    headers = {'subject': message.subject or '', 'sender': message.sender or '', 'to': message.to or '', 'date': str(message.date)}
    body = (message.body or '').replace('\r\n', '\n')
    email_id = hashlib.sha256(json.dumps({**headers, 'body': body}, sort_keys=True).encode()).hexdigest()
    evidence_path = claim_folder.joinpath(*PureWindowsPath(source.split('::', 1)[0]).parts[1:])
    item = emails.setdefault(email_id, {'filename': PureWindowsPath(source.rsplit('::', 1)[-1]).name, 'source_path': str(evidence_path.resolve()), 'content': '# ' + headers['subject'] + '\n\n' + '\n'.join(f'{key}: {value}' for key, value in headers.items()) + '\n\n' + body, 'source_files': [], 'parent_emails': []})
    already_read = bool(item['source_files'])
    item['source_files'].append(source)
    if parent_email and parent_email not in item['parent_emails']:
        item['parent_emails'].append(parent_email)
    if already_read:
        return
    for number, attachment in enumerate(message.attachments):
        name = PureWindowsPath(str(attachment.getFilename() or f'attachment-{number}')).name
        data = attachment.data
        if isinstance(data, MSGFile):
            ReadMessage(data, f'{source}::{name}', email_id)
        elif isinstance(data, bytes):
            AddDocument(data, name, f'{source}::{name}', email_id)
        else:
            raise TypeError(f'Unsupported attachment data: {source}::{name}')

In [6]:
print('-' * 72)
print('Step 2: Extracting email text and attachments')
for path in files:
    source = f'{claim_id}/{path.relative_to(claim_folder).as_posix()}'
    if path.suffix.lower() in {'.pdf', '.jpg', '.jpeg', '.png'}:
        AddDocument(path.read_bytes(), path.name, source)
    elif path.suffix.lower() != '.msg':
        skipped.append({'source': source, 'reason': 'outside the four-format demo'})
for path in files:
    if path.suffix.lower() == '.msg':
        with extract_msg.openMsg(str(path)) as message:
            ReadMessage(message, f'{claim_id}/{path.relative_to(claim_folder).as_posix()}')

document_df = pd.DataFrame([{'document_hash': key, 'filename': item['filename'], 'source_files': item['source_files'], 'parent_emails': item['parent_emails'], 'occurrences': len(item['source_files'])} for key, item in documents.items()])
ShowPreview(document_df)
ShowPreview(pd.DataFrame([{'filename': item['filename'], 'content': item['content']} for item in emails.values()]))
if skipped:
    ShowPreview(pd.DataFrame(skipped))
print('Unique documents for CU:', len(documents), '| Unique emails:', len(emails))
print('Duplicate visual occurrences avoided:', sum(len(item['source_files']) - 1 for item in documents.values()))

------------------------------------------------------------------------
Step 2: Extracting email text and attachments


,document_hash,filename,source_files,parent_emails,occurrences
0,29c778e0261eb6326e50990e1440ef0ecc92ed63332ae845c6a6f93e6a5943ea,claim-form.pdf,"['claim-001/claim-form.pdf', 'claim-001/01-notification.msg::claim-form.pdf']",['8e65cd228a775af62c28ae1164216cbd53e71f8a2fc1f65aef3e7e5ad0d1b84a'],2
1,dced4e48ca24c1df7a8d417b8ce365a8561393d1d709778bfbc456d43633aee8,damage-01.jpg,"['claim-001/damage-01.jpg', 'claim-001/01-notification.msg::damage-01.jpg']",['8e65cd228a775af62c28ae1164216cbd53e71f8a2fc1f65aef3e7e5ad0d1b84a'],2
2,b71b52334d5793e1588b1febcea4746f16af164e819e897903b9da00165f43a4,damage-02.jpg,['claim-001/damage-02.jpg'],[],1
3,8d6710737997401b08cd47802f20e7d20d053925047897e4a286532d1f50ceba,incident-report-scan.pdf,['claim-001/incident-report-scan.pdf'],[],1
4,fd9e9580fd9649ab746307d42c593d43bbbabb0153d04158afd4a315bf4d976b,inspection.pdf,"['claim-001/inspection.pdf', 'claim-001/02-inspection-and-attachments.msg::inspection-review-copy.pdf']",['e6d9d56b2213a0fda54d4f9eb30cc58a99b5945960b705ff611db5f1fcc9de54'],2


,filename,content
0,01-notification.msg,# AV-26-004181 - notification and site photograph\n\nsubject: AV-26-004181 - notification and site photograph\nsender: Morgan Hale\nto: claims@example.inval...
1,02-inspection-and-attachments.msg,"# AV-26-004181 - illustrated report, authority and access note\n\nsubject: AV-26-004181 - illustrated report, authority and access note\nsender: Meridian Fi..."
2,03-adjuster-follow-up.msg,# RE: AV-26-004181 - outstanding evidence\n\nsubject: RE: AV-26-004181 - outstanding evidence\nsender: Alex Rowan\nto: claims@example.invalid <None>\ndate: ...


Unique documents for CU: 9 | Unique emails: 3
Duplicate visual occurrences avoided: 3


## Review document and image routing
PDFs use `prebuilt-documentSearch`, including their embedded figures. Photos and diagrams use `prebuilt-imageSearch`, including image attachments. A file extension does not establish an image's purpose: add reviewed image-format scans to `document_image_names` to retain document/OCR analysis. Leave it empty for these fixtures; never exclude other images just because they are email attachments.

In [7]:
import sys
if Path(sys.executable).resolve() != (ROOT / '.venv' / 'Scripts' / 'python.exe').resolve():
    raise RuntimeError('Select the repository .venv kernel before running this cell.')
document_image_names = set()
image_names = {item['filename'] for item in documents.values() if item['source_type'] == 'image'}
if document_image_names - image_names:
    raise ValueError(f'Unknown image scan overrides: {document_image_names - image_names}')
analyzer_routes = {key: document_analyzer_id if item['source_type'] == 'document' or item['filename'] in document_image_names else image_analyzer_id for key, item in documents.items()}
ShowPreview(pd.DataFrame([{'filename': item['filename'], 'analyzer_id': analyzer_routes[key], 'parent_emails': item['parent_emails']} for key, item in documents.items() if item['source_type'] == 'image']))
print('Documents:', sum(route == document_analyzer_id for route in analyzer_routes.values()), '| Visual images:', sum(route == image_analyzer_id for route in analyzer_routes.values()))
print('Kernel:', Path(sys.executable).relative_to(ROOT))

,filename,analyzer_id,parent_emails
0,damage-01.jpg,prebuilt-imageSearch,['8e65cd228a775af62c28ae1164216cbd53e71f8a2fc1f65aef3e7e5ad0d1b84a']
1,damage-02.jpg,prebuilt-imageSearch,[]
2,survey-chart.png,prebuilt-imageSearch,[]
3,site-access.png,prebuilt-imageSearch,['e6d9d56b2213a0fda54d4f9eb30cc58a99b5945960b705ff611db5f1fcc9de54']


Documents: 5 | Visual images: 4
Kernel: .venv\Scripts\python.exe


## Step 3: Extract with the out-of-box CU analyzer
Document analysis supplies native Markdown and embedded-figure descriptions. Image analysis supplies a native `Summary` field, not document Markdown. Use the reviewed route for each unique file; do not separately submit PDF crops. Cache keys include document bytes, the actual analyzer and API. Use a new output folder after intentionally changing cloud resources or model configuration.

In [8]:
print('-' * 72)
print('Step 3: Content Understanding extraction')
cu_results = {}
cu_summary = []
for document_hash, item in documents.items():
    start = time.time()
    analyzer_id = analyzer_routes[document_hash]
    result_file = results_folder / f'{analyzer_id}_{api_version}_{document_hash}.json'
    cached = result_file.exists()
    if cached:
        result = json.loads(result_file.read_text(encoding='utf-8'))
    else:
        poller = cu_client.begin_analyze(analyzer_id=analyzer_id, inputs=[AnalysisInput(data=item['data'])])
        result = poller.result().as_dict()
        if not result.get('contents'):
            raise RuntimeError(f"CU returned no content for {item['filename']}.")
        result_file.write_text(json.dumps(result, indent=2, cls=AzureJSONEncoder), encoding='utf-8')
    if not result.get('contents') or result.get('analyzerId') != analyzer_id or result.get('apiVersion') != api_version:
        raise RuntimeError(f"Invalid or incompatible CU result for {item['filename']}; inspect {result_file}.")
    cu_results[document_hash] = result
    cu_summary.append({'filename': item['filename'], 'analyzer_id': analyzer_id, 'cached': cached, 'seconds': round(time.time() - start, 2)})
ShowPreview(pd.DataFrame(cu_summary))
print('Documents analyzed or loaded from cache:', len(cu_results))

------------------------------------------------------------------------
Step 3: Content Understanding extraction


,filename,analyzer_id,cached,seconds
0,claim-form.pdf,prebuilt-documentSearch,False,9.82
1,damage-01.jpg,prebuilt-imageSearch,False,6.59
2,damage-02.jpg,prebuilt-imageSearch,False,6.89
3,incident-report-scan.pdf,prebuilt-documentSearch,False,6.59
4,inspection.pdf,prebuilt-documentSearch,False,13.31


Documents analyzed or loaded from cache: 9


## Step 4: Preview native CU output
Save all native document Markdown and image summaries. Display one short document example, one figure description and compact image-summary previews. Nothing is reconstructed or verbalized by a second custom extraction prompt.

In [9]:
print('-' * 72)
print('Step 4: Native Markdown and figure example')
for document_hash, result in cu_results.items():
    for number, content in enumerate(result['contents']):
        native_text = (content.get('fields') or {}).get('Summary', {}).get('valueString', '') if analyzer_routes[document_hash] == image_analyzer_id else content.get('markdown') or ''
        (output_folder / f'{document_hash}_{number}.md').write_text(native_text, encoding='utf-8')
example_hash = next((key for key, result in cu_results.items() if any(content.get('figures') for content in result['contents'])), next(iter(cu_results)))
example_contents = cu_results[example_hash]['contents']
example = next((content for content in example_contents if content.get('figures')), example_contents[0])
print('Example:', documents[example_hash]['filename'])
example_text = example.get('markdown') or (example.get('fields') or {}).get('Summary', {}).get('valueString', '')
display(Markdown(example_text[:1200]))
print('Pages:', len(example.get('pages') or []), '| Tables:', len(example.get('tables') or []), '| Figures:', len(example.get('figures') or []))
ShowPreview(pd.DataFrame([{'id': figure.get('id'), 'description': figure.get('description')} for figure in (example.get('figures') or [])[:1]]))
ShowPreview(pd.DataFrame([{'filename': documents[key]['filename'], 'image_observation': (content.get('fields') or {}).get('Summary', {}).get('valueString', '')} for key, result in cu_results.items() if analyzer_routes[key] == image_analyzer_id for content in result['contents']]))

------------------------------------------------------------------------
Step 4: Native Markdown and figure example
Example: inspection.pdf


<!-- PageHeader: MERIDIAN FIELD ADJUSTING | FICTIONAL PRACTICE -->

Fictional demonstration material | Not an insurance contract


# Inspection report

AV-26-004181 | Site visit 2026-09-14 | Report revision 1

INSURED

Cedar Quay Warehousing LLC

SITE

Building 4, East Quay Industrial Park, Fairhaven

REPORTED EVENT

Wind damage and subsequent water entry at the east roof corner

INSPECTING ADJUSTER

Alex Rowan


## 1. Reported circumstances

The site manager reported displaced roof sheets after overnight high winds. Water was
discovered beside pallet storage at the east wall when the building opened. Staff isolated the
affected aisle and arranged temporary weather protection. No business-interruption amount has
been agreed.


## 2. Observations and limitations

The east roof corner has displaced metal sheets and temporary blue weather protection.
Interior ceiling insulation is locally stained. Cartons beside the east wall show moisture
exposure. These observations alone do not establish when the opening formed or whether all
stock was damaged in this event.

See the embedded annotated photo exhibit and case-specific location sketch. Both are original
synthetic scenario exhibits; t

Pages: 4 | Tables: 1 | Figures: 2


,id,description
0,3.1,Two-panel photo labeled “A” (left) and “B” (right).\nLeft panel (A): Exterior corner of a light-gray metal-clad building with a roller door; roof edge shows...


,filename,image_observation
0,damage-01.jpg,"The image shows the corner of a light-gray metal building with significant roof damage: several corrugated metal roof panels are bent upward and torn, expos..."
1,damage-02.jpg,The image shows the interior of a metal storage shed or warehouse with corrugated metal walls and a translucent corrugated roof supported by a metal frame. ...
2,survey-chart.png,The image is a bar chart titled “SITE MOISTURE SURVEY – SAMPLE READINGS” with reference “AV-26-004181” and date “2026-09-14.” It shows three zones with mois...
3,site-access.png,"The image shows a “SITE ACCESS / APPOINTMENT NOTE” with reference number AV-26-004181. It lists a reinspection scheduled for 2026-09-20 at 10:00, instructs ..."


## Step 5: Extract Markdown output into a DataFrame
Split native document Markdown by headings. Each native image summary is an image-description row. Keep native figure descriptions as separately citable rows, even when they overlap document Markdown. References are file-level, with native figure sources when available; there is no separate page-number field. Do not turn visual observations into claims decisions.

In [10]:
def MdFormatting(markdown):
    text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[('#', 'Title'), ('##', 'Header 1')], strip_headers=False)
    return pd.DataFrame([{'title': chunk.metadata.get('Title', ''), 'header': chunk.metadata.get('Header 1', ''), 'content': chunk.page_content} for chunk in text_splitter.split_text(markdown)], columns=['title', 'header', 'content'])

print('-' * 72)
print('Step 5: Preparing claim evidence rows')
start = time.time()
rows = []
for document_hash, result in cu_results.items():
    item = documents[document_hash]
    evidence = {'document_hash': document_hash, 'filename': item['filename'], 'source_path': item['source_path'], 'source_type': item['source_type'], 'source_files': item['source_files'], 'parent_emails': item['parent_emails'], 'analyzer_id': analyzer_routes[document_hash], 'analyzer_api_version': result['apiVersion']}
    previous_rows = len(rows)
    for content in result['contents']:
        if evidence['analyzer_id'] == image_analyzer_id:
            observation = (content.get('fields') or {}).get('Summary', {}).get('valueString', '')
            if not observation.strip():
                raise RuntimeError(f"No native image summary for {item['filename']}; do not silently omit this evidence.")
            rows.append({**evidence, 'title': item['filename'], 'header': 'Visual observation; not a causation or coverage finding', 'content': observation, 'content_kind': 'image_description', 'figure_id': '', 'source_region': ''})
            continue
        markdown = content.get('markdown') or ''
        for chunk in MdFormatting(markdown).to_dict(orient='records'):
            rows.append({**evidence, **chunk, 'content_kind': 'document_text', 'figure_id': '', 'source_region': content.get('source') or ''})
        for figure in content.get('figures') or []:
            description = figure.get('description')
            if description:
                rows.append({**evidence, 'title': item['filename'], 'header': f"CU figure {figure['id']} description", 'content': description, 'content_kind': 'figure_description', 'figure_id': figure['id'], 'source_region': figure.get('source') or ''})
    if len(rows) == previous_rows:
        raise RuntimeError(f"No searchable evidence extracted from {item['filename']} by {evidence['analyzer_id']}; review its route and raw result.")
for email_id, item in emails.items():
    for chunk in MdFormatting(item['content']).to_dict(orient='records'):
        rows.append({**chunk, 'document_hash': email_id, 'filename': item['filename'], 'source_path': item['source_path'], 'source_type': 'email', 'content_kind': 'email_text', 'figure_id': '', 'source_region': '', 'source_files': item['source_files'], 'parent_emails': item['parent_emails'], 'analyzer_id': 'local-msg', 'analyzer_api_version': ''})
if not rows:
    raise RuntimeError('No rows were extracted; do not generate or upload an empty dataset.')
df = pd.DataFrame(rows)
df['title'] = df.apply(lambda row: row['title'] or row['filename'], axis=1)
print('Chunks:', len(df), '| Seconds:', round(time.time() - start, 2))
ShowPreview(df[['filename', 'content_kind', 'title', 'content']])
print('Evidence kinds:', df['content_kind'].value_counts().to_dict())

------------------------------------------------------------------------
Step 5: Preparing claim evidence rows
Chunks: 33 | Seconds: 0.03


,filename,content_kind,title,content
0,claim-form.pdf,document_text,claim-form.pdf,<!-- PageHeader: ASTER VALE | CLAIMS SERVICES --> \nFictional demonstration material | Not an insurance contract
1,claim-form.pdf,document_text,First notification of loss,# First notification of loss \nAV-26-004181 | Cedar Quay Warehousing LLC \nCLAIM REFERENCE\nAV-26-004181 \nPOLICY REFERENCE\nAV-CP-260101 \nINSURED / RE...
2,claim-form.pdf,document_text,First notification of loss,## Account supplied at notification \nThe site manager reported displaced roof sheets after overnight high winds. Water was\ndiscovered beside pallet stora...
3,damage-01.jpg,image_description,damage-01.jpg,"The image shows the corner of a light-gray metal building with significant roof damage: several corrugated metal roof panels are bent upward and torn, expos..."
4,damage-02.jpg,image_description,damage-02.jpg,The image shows the interior of a metal storage shed or warehouse with corrugated metal walls and a translucent corrugated roof supported by a metal frame. ...


Evidence kinds: {'document_text': 24, 'image_description': 4, 'email_text': 3, 'figure_description': 2}


## Step 6: Print character and token counts
Embed title, section, evidence kind and content together once per row. Count the exact embedding input, not just its content. Oversized inputs must be split explicitly, never truncated.

In [11]:
encoding = tiktoken.get_encoding('cl100k_base')
def NumberTokens(text):
    return len(encoding.encode(text, disallowed_special=()))

print('-' * 72)
print('Step 6: Counting characters and tokens')
df['embedding_text'] = df.apply(lambda row: '\n\n'.join([f"Title: {row['title']}", f"Section: {row['header']}", f"Evidence kind: {row['content_kind']}", row['content']]), axis=1)
df['char_len'] = df['embedding_text'].apply(len)
df['token_len'] = df['embedding_text'].apply(NumberTokens)
ShowPreview(df[['filename', 'title', 'char_len', 'token_len']])
print('Chunks:', len(df), '| Largest chunk:', df['token_len'].max(), 'tokens')
if (df['token_len'] > 8000).any():
    raise ValueError('A contextual embedding input exceeds 8000 tokens. Split it before continuing.')

------------------------------------------------------------------------
Step 6: Counting characters and tokens


,filename,title,char_len,token_len
0,claim-form.pdf,claim-form.pdf,176,39
1,claim-form.pdf,First notification of loss,409,121
2,claim-form.pdf,First notification of loss,680,139
3,damage-01.jpg,damage-01.jpg,501,104
4,damage-02.jpg,damage-02.jpg,586,123


Chunks: 33 | Largest chunk: 389 tokens


## Embedding client and direct GenerateVector function
Use the same Foundry account and DefaultAzureCredential as CU. The OpenAI v1 endpoint is derived from the CU endpoint; no separate endpoint or API key is needed. Titles and content are vectorized separately in the following paid-call cells.

In [12]:
from openai import OpenAI
import os

foundry_openai_endpoint = (os.getenv('AZURE_OPENAI_ENDPOINT') or os.environ['APP_CONTENT_UNDERSTANDING_ENDPOINT'].replace('.cognitiveservices.azure.com', '.openai.azure.com').replace('.services.ai.azure.com', '.openai.azure.com')).rstrip('/')
if not azure_openai_embedding_deployment:
    raise ValueError('Set EMBEDDING_MODEL_DEPLOYMENT to an existing embedding deployment in the CU Foundry account.')
# The SDK's api_key parameter receives an Entra token provider, not an API key.
aoai_client = OpenAI(
    base_url=f'{foundry_openai_endpoint}/openai/v1/',
    api_key=get_bearer_token_provider(credential, 'https://ai.azure.com/.default'),
)

def GenerateVector(text):
    if NumberTokens(text) > 8000:
        raise ValueError('Text is too long to embed; split it rather than truncate it.')
    response = aoai_client.embeddings.create(input=text, model=azure_openai_embedding_deployment, dimensions=azure_openai_model_dimensions)
    vector = response.data[0].embedding
    if len(vector) != azure_openai_model_dimensions:
        raise ValueError('Embedding dimensions do not match the configured index dimensions.')
    return vector

print('Entra ID | Same CU account | Deployment:', azure_openai_embedding_deployment, '| Dimensions:', azure_openai_model_dimensions)

Entra ID | Same CU account | Deployment: text-embedding-3-large | Dimensions: 3072


In [13]:
print('-' * 72)
print('Step 7: Reviewing contextual embedding inputs')
if df['content'].str.strip().eq('').any() or not df['token_len'].between(1, 8000).all():
    raise ValueError('Empty or oversized evidence cannot be embedded.')
ShowPreview(df[['filename', 'content_kind', 'embedding_text', 'token_len']])
print('One embedding per row:', len(df), '| No separate title vectors')

------------------------------------------------------------------------
Step 7: Reviewing contextual embedding inputs


,filename,content_kind,embedding_text,token_len
0,claim-form.pdf,document_text,Title: claim-form.pdf\n\nSection: \n\nEvidence kind: document_text\n\n<!-- PageHeader: ASTER VALE | CLAIMS SERVICES --> \nFictional demonstration material ...,39
1,claim-form.pdf,document_text,Title: First notification of loss\n\nSection: \n\nEvidence kind: document_text\n\n# First notification of loss \nAV-26-004181 | Cedar Quay Warehousing LLC ...,121
2,claim-form.pdf,document_text,Title: First notification of loss\n\nSection: Account supplied at notification\n\nEvidence kind: document_text\n\n## Account supplied at notification \nThe...,139
3,damage-01.jpg,image_description,Title: damage-01.jpg\n\nSection: Visual observation; not a causation or coverage finding\n\nEvidence kind: image_description\n\nThe image shows the corner o...,104
4,damage-02.jpg,image_description,Title: damage-02.jpg\n\nSection: Visual observation; not a causation or coverage finding\n\nEvidence kind: image_description\n\nThe image shows the interior...,123


One embedding per row: 33 | No separate title vectors


In [14]:
print('-' * 72)
print('Step 8: Vectorizing content')
start = time.time()
df['content_vector'] = df['embedding_text'].apply(GenerateVector)
print('Chunks embedded:', len(df), '| Seconds:', round(time.time() - start, 2))
ShowPreview(df[['filename', 'content_vector']])

------------------------------------------------------------------------
Step 8: Vectorizing content
Chunks embedded: 33 | Seconds: 18.19


,filename,content_vector
0,claim-form.pdf,3072 dimensions
1,claim-form.pdf,3072 dimensions
2,claim-form.pdf,3072 dimensions
3,damage-01.jpg,3072 dimensions
4,damage-02.jpg,3072 dimensions


## Step 9: Add the index metadata and deterministic IDs
The claim ID comes from the selected folder, not image interpretation. Chunk numbers are local to each document; preserve original aliases, parent emails and local evidence paths. Keep disagreements in their original evidence rows rather than inventing authoritative claim facts.

In [15]:
print('-' * 72)
print('Step 9: Adding index metadata and IDs')
df['claim_id'] = claim_id
df['chunk_id'] = df.groupby('document_hash', sort=False).cumcount().add(1)
df['id'] = df.apply(lambda row: hashlib.sha256(f"{claim_id}:{row['document_hash']}:{row['chunk_id']}".encode()).hexdigest(), axis=1)
ShowPreview(df[['claim_id', 'filename', 'source_type', 'content_kind', 'chunk_id', 'figure_id', 'analyzer_id']])

------------------------------------------------------------------------
Step 9: Adding index metadata and IDs


,claim_id,filename,source_type,content_kind,chunk_id,figure_id,analyzer_id
0,claim-001,claim-form.pdf,document,document_text,1,,prebuilt-documentSearch
1,claim-001,claim-form.pdf,document,document_text,2,,prebuilt-documentSearch
2,claim-001,claim-form.pdf,document,document_text,3,,prebuilt-documentSearch
3,claim-001,damage-01.jpg,image,image_description,1,,prebuilt-imageSearch
4,claim-001,damage-02.jpg,image,image_description,1,,prebuilt-imageSearch


## Step 10: Preview the DataFrame that will be loaded to the index
Keep every evidence row and its full contextual content vector in index_df. Display only five rows, shortened text and vector dimensions. Notebooks 2 and 3 use exactly the same claims-evidence columns. Native document summaries remain in raw CU results rather than being repeated on every indexed chunk.

In [16]:
print('-' * 72)
print('Step 10: Final index-ready DataFrame')
index_columns = ['id', 'claim_id', 'document_hash', 'chunk_id', 'title', 'header', 'content', 'source_type', 'content_kind', 'filename', 'source_path', 'source_files', 'parent_emails', 'figure_id', 'source_region', 'analyzer_id', 'analyzer_api_version', 'content_vector']
index_df = df[index_columns].copy()
ShowPreview(index_df)
print('Rows ready to load:', len(index_df))
print('Columns:', index_df.columns.tolist())
print('Preview only; full values remain in index_df.')

------------------------------------------------------------------------
Step 10: Final index-ready DataFrame


,id,claim_id,document_hash,chunk_id,title,header,content,...,source_files,parent_emails,figure_id,source_region,analyzer_id,analyzer_api_version,content_vector
0,49d92297125cfe67adbf6889ba47af85921ba59d272fb651c1b6546ec961a6fd,claim-001,29c778e0261eb6326e50990e1440ef0ecc92ed63332ae845c6a6f93e6a5943ea,1,claim-form.pdf,,<!-- PageHeader: ASTER VALE | CLAIMS SERVICES --> \nFictional demonstration material | Not an insurance contract,...,"['claim-001/claim-form.pdf', 'claim-001/01-notification.msg::claim-form.pdf']",['8e65cd228a775af62c28ae1164216cbd53e71f8a2fc1f65aef3e7e5ad0d1b84a'],,,prebuilt-documentSearch,2025-11-01,3072 dimensions
1,75d4be7842008c2e921dd2babb074696d168d7ad0fa6ccd901c43445eee55ba2,claim-001,29c778e0261eb6326e50990e1440ef0ecc92ed63332ae845c6a6f93e6a5943ea,2,First notification of loss,,# First notification of loss \nAV-26-004181 | Cedar Quay Warehousing LLC \nCLAIM REFERENCE\nAV-26-004181 \nPOLICY REFERENCE\nAV-CP-260101 \nINSURED / RE...,...,"['claim-001/claim-form.pdf', 'claim-001/01-notification.msg::claim-form.pdf']",['8e65cd228a775af62c28ae1164216cbd53e71f8a2fc1f65aef3e7e5ad0d1b84a'],,,prebuilt-documentSearch,2025-11-01,3072 dimensions
2,dac5283c434b69f690efa44f2df19a3059b73ec45e18c1703164a2408e9f933c,claim-001,29c778e0261eb6326e50990e1440ef0ecc92ed63332ae845c6a6f93e6a5943ea,3,First notification of loss,Account supplied at notification,## Account supplied at notification \nThe site manager reported displaced roof sheets after overnight high winds. Water was\ndiscovered beside pallet stora...,...,"['claim-001/claim-form.pdf', 'claim-001/01-notification.msg::claim-form.pdf']",['8e65cd228a775af62c28ae1164216cbd53e71f8a2fc1f65aef3e7e5ad0d1b84a'],,,prebuilt-documentSearch,2025-11-01,3072 dimensions
3,293119ce5a2c1a2755bb348da4a2aea2feaca39c59291ed4ac758866f85c21f2,claim-001,dced4e48ca24c1df7a8d417b8ce365a8561393d1d709778bfbc456d43633aee8,1,damage-01.jpg,Visual observation; not a causation or coverage finding,"The image shows the corner of a light-gray metal building with significant roof damage: several corrugated metal roof panels are bent upward and torn, expos...",...,"['claim-001/damage-01.jpg', 'claim-001/01-notification.msg::damage-01.jpg']",['8e65cd228a775af62c28ae1164216cbd53e71f8a2fc1f65aef3e7e5ad0d1b84a'],,,prebuilt-imageSearch,2025-11-01,3072 dimensions
4,d6202cc1ff3ad071b403f94c81b0924937c983798cc566a3948418f0fb34eef7,claim-001,b71b52334d5793e1588b1febcea4746f16af164e819e897903b9da00165f43a4,1,damage-02.jpg,Visual observation; not a causation or coverage finding,The image shows the interior of a metal storage shed or warehouse with corrugated metal walls and a translucent corrugated roof supported by a metal frame. ...,...,['claim-001/damage-02.jpg'],[],,,prebuilt-imageSearch,2025-11-01,3072 dimensions


Rows ready to load: 33
Columns: ['id', 'claim_id', 'document_hash', 'chunk_id', 'title', 'header', 'content', 'source_type', 'content_kind', 'filename', 'source_path', 'source_files', 'parent_emails', 'figure_id', 'source_region', 'analyzer_id', 'analyzer_api_version', 'content_vector']
Preview only; full values remain in index_df.


## Step 11: Save processed data for the separate loading notebook
Verify that every unique input and email remains represented, with usable provenance. Save the new schema as `processed-evidence-v1.json`; the old `processed.json` is left untouched. Notebook 3 reads this file directly; upload retries do not repeat CU or embeddings.

In [17]:
print('-' * 72)
print('Step 11: Saving processed index records')
if set(index_df['document_hash']) != set(documents) | set(emails) or not index_df['id'].is_unique:
    raise ValueError('Evidence is missing or chunk IDs are duplicated; nothing was saved.')
if not index_df['source_path'].map(lambda value: Path(value).is_file()).all():
    raise ValueError('An evidence source path does not exist; nothing was saved.')
if any(parent not in emails for parents in index_df['parent_emails'] for parent in parents):
    raise ValueError('An attachment has an unknown parent email; nothing was saved.')
for key, result in cu_results.items():
    if analyzer_routes[key] == image_analyzer_id:
        native_observations = [(content.get('fields') or {}).get('Summary', {}).get('valueString', '') for content in result['contents']]
        if index_df.loc[index_df['document_hash'].eq(key), 'content'].tolist() != native_observations:
            raise ValueError('A native image observation was omitted or changed; nothing was saved.')
processed_file = output_folder / 'processed-evidence-v1.json'
index_df.to_json(processed_file, orient='records', indent=2, force_ascii=False, double_precision=15)
print('Saved', len(index_df), 'full records to:', processed_file)
print('Next: run 02_create_index.ipynb, then 03_populate_index.ipynb.')
cu_client.close()
aoai_client.close()
credential.close()

------------------------------------------------------------------------
Step 11: Saving processed index records
Saved 33 full records to: c:\Users\jomedin\Documents\azure-ai-foundry-insurance\output\claims\ais-foundry-eval-dev-02\claim-001\processed-evidence-v1.json
Next: run 02_create_index.ipynb, then 03_populate_index.ipynb.
